# Train Logistic Regression with Minibatches

This tutorial shows how to train an incremental logistic-regression classifier
from Atlas expression minibatches. It is the simplest supervised example of
using scAtlasPy as a data-access layer for custom model development.

The example uses scikit-learn's `SGDClassifier` with logistic loss. Expression
data are read in dense minibatches, while the full cell-by-gene matrix remains
in the Atlas.

By the end of this tutorial, you will be able to:

- define a labeled training population in `obs`;
- align labels with a deterministic expression stream;
- train a classifier incrementally with `partial_fit()`;
- make several training passes without loading the full matrix;
- evaluate the training pipeline in a second streaming pass;
- save the fitted model for later full-Atlas prediction.

## Before You Begin

This tutorial assumes that:

- quality control and preprocessing have been completed;
- `obs.cell_type_manual` contains the target labels;
- scaled expression values are available in `data_scale`;
- the selected dense minibatches fit in memory.

Open the existing Atlas:



In [ ]:
import numpy as np
import scatlaspy as sap

atlas = sap.Atlas(
    "./data/pbmc_demo.sasql",
    db_memory_limit="8GB",
)



```{note}
This tutorial uses known cell-type labels only as an example supervised target.
The same pattern can be applied to another categorical outcome stored in
`obs`.
```

## 1. Define the Labeled Training Population

The expression stream and label vector must contain exactly the same cells in
the same order.

First, create a persistent Boolean column identifying cells that:

- pass the existing cell filter;
- have a non-missing training label.



In [ ]:
atlas.execute_sql("""
    ALTER TABLE obs
    ADD COLUMN IF NOT EXISTS model_labeled_cells BOOLEAN
""")

atlas.execute_sql("""
    UPDATE obs
    SET model_labeled_cells =
        COALESCE(filter_cells, FALSE)
        AND cell_type_manual IS NOT NULL
""")



Inspect the number of labeled cells in each class:



In [ ]:
atlas.query("""
    SELECT
        cell_type_manual,
        COUNT(*) AS n_cells
    FROM obs
    WHERE model_labeled_cells
    GROUP BY cell_type_manual
    ORDER BY n_cells DESC
""")



Confirm that the selected population contains at least two classes and that
each class contains enough cells for the intended analysis.

```{important}
Do not build the read index from all filtered cells and then remove unlabeled
rows only from the label table. That would cause the expression stream and
labels to become misaligned.
```

## 2. Build the Training Read Index

Build a read index using the labeled-cell column:



In [ ]:
atlas.build_read_index(
    cell_condition="model_labeled_cells",
    gene_condition="filter_genes",
    use_hvg=True,
    use_data="data_scale",
)



The model input now consists of:

- cells selected by `model_labeled_cells`;
- genes selected by `filter_genes`;
- genes marked as highly variable;
- scaled expression values stored in `data_scale`.

```{important}
The trained model depends on the exact gene set, gene order, and expression
representation defined by this read index. Record this information alongside
the fitted model.
```

Rebuilding the read index changes the active data view used by subsequent
streaming operations. It does not change the stored expression values, but
later computations should use a read index appropriate for their own purpose.

## 3. Read Labels in Streaming Order

In `single-pass` mode, expression minibatches follow the current
`filter_cell_id` order. Retrieve labels in that same order:



In [ ]:
label_df = atlas.query("""
    SELECT
        filter_cell_id,
        cell_type_manual
    FROM obs
    WHERE filter_cell_id IS NOT NULL
    ORDER BY filter_cell_id
""")



Verify that no selected cell has a missing label:



In [ ]:
if label_df.empty:
    raise ValueError(
        "The current read index contains no labeled cells."
    )

if label_df["cell_type_manual"].isna().any():
    raise ValueError(
        "The current read index contains cells without training labels."
    )

if label_df["filter_cell_id"].duplicated().any():
    raise ValueError(
        "The current read index contains duplicated filter_cell_id values."
    )



Encode text labels as consecutive integers:



In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

labels = label_encoder.fit_transform(
    label_df["cell_type_manual"].to_numpy()
)

classes = np.arange(len(label_encoder.classes_))

print(f"Training cells: {len(labels):,}")
print(f"Classes: {len(classes):,}")
print(label_encoder.classes_)



Check that classification is possible:



In [ ]:
if len(classes) < 2:
    raise ValueError(
        "At least two label classes are required for classification."
    )



The encoded labels use less memory and can later be converted back to their
original names with `label_encoder.inverse_transform()`.

## 4. Configure the Classifier

Create an incremental linear classifier with logistic loss:



In [ ]:
from sklearn.linear_model import SGDClassifier

model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    random_state=42,
)



`partial_fit()` updates the existing model from one minibatch at a time. The
complete class list must be supplied during the first update because the first
minibatch may not contain every class.

The scaled input used in this tutorial is appropriate for gradient-based
optimization. If another expression field is used, consider whether its scale
is suitable for the selected model.

## 5. Train in Minibatches

Set the minibatch size and number of passes:



In [ ]:
batch_size = 2048
n_epochs = 3



Each epoch creates a new deterministic single-pass stream. This preserves label
alignment while allowing the model to revisit the selected cells.



In [ ]:
first_update = True

for epoch in range(1, n_epochs + 1):
    offset = 0

    for batch_id, X_batch in enumerate(
        atlas.get_minibatch_dense(
            pass_mode="single-pass",
            batch_size=batch_size,
        ),
        start=1,
    ):
        X_batch = np.asarray(
            X_batch,
            dtype=np.float32,
        )

        if X_batch.ndim != 2:
            raise ValueError(
                "Each expression minibatch must be a two-dimensional matrix."
            )

        n_cells = X_batch.shape[0]
        y_batch = labels[offset : offset + n_cells]

        if len(y_batch) != n_cells:
            raise ValueError(
                "The label vector is shorter than the expression stream."
            )

        if first_update:
            model.partial_fit(
                X_batch,
                y_batch,
                classes=classes,
            )
            first_update = False
        else:
            model.partial_fit(
                X_batch,
                y_batch,
            )

        offset += n_cells

        if batch_id == 1 or batch_id % 50 == 0:
            print(
                f"Epoch {epoch}/{n_epochs}, "
                f"batch {batch_id}: "
                f"processed {offset:,} cells"
            )

    if offset != len(labels):
        raise ValueError(
            "The number of streamed cells does not match the label vector."
        )

    print(
        f"Completed epoch {epoch}/{n_epochs}: "
        f"{offset:,} cells"
    )



Only the current expression minibatch and the model parameters need to reside
in memory. The label vector is stored in memory, but it is much smaller than
the full expression matrix.

```{note}
Repeated single-pass traversal preserves the same global cell order between
epochs. This example is intended as a clear baseline rather than a complete
training strategy.

For order-sensitive optimization, consider how cells were ordered when the
Atlas was imported and whether additional joint shuffling of expression rows
and labels is needed.
```

## 6. Check the Training Pipeline

Run another single pass and calculate predictions without retaining all
expression data or predictions:



In [ ]:
correct = 0
total = 0
offset = 0

for X_batch in atlas.get_minibatch_dense(
    pass_mode="single-pass",
    batch_size=batch_size,
):
    X_batch = np.asarray(
        X_batch,
        dtype=np.float32,
    )

    n_cells = X_batch.shape[0]
    y_batch = labels[offset : offset + n_cells]

    if len(y_batch) != n_cells:
        raise ValueError(
            "The label vector is shorter than the evaluation stream."
        )

    pred = model.predict(X_batch)

    correct += int(
        np.count_nonzero(pred == y_batch)
    )
    total += n_cells
    offset += n_cells

if offset != len(labels):
    raise ValueError(
        "The number of evaluated cells does not match the label vector."
    )

if total == 0:
    raise ValueError(
        "The evaluation stream contained no cells."
    )

training_accuracy = correct / total

print(
    f"Training-set accuracy: "
    f"{training_accuracy:.3f}"
)



```{warning}
Training-set accuracy checks that expression values, labels, and model outputs
are aligned correctly. It is not an unbiased estimate of predictive
performance because the same cells were used for fitting and evaluation.
```

## 7. Evaluate on Held-out Cells

For a meaningful evaluation, define non-overlapping training and test
populations in `obs`.

For example, create Boolean columns such as:

```text
model_train_cells
model_test_cells
```

Then:

1. build the read index from `model_train_cells`;
2. retrieve its labels in `filter_cell_id` order;
3. fit the classifier;
4. rebuild the read index from `model_test_cells`;
5. retrieve the test labels in the new `filter_cell_id` order;
6. evaluate the model with a new single-pass stream.

The same alignment rule applies to both populations: labels must be queried
after building the corresponding read index and ordered by its
`filter_cell_id`.

```{tip}
Choose a validation split that reflects the intended application. For example,
holding out complete donors, samples, studies, or technologies can be more
informative than randomly holding out individual cells when the goal is
generalization across biological or technical contexts.
```

Accuracy may also be misleading when classes are highly imbalanced. Consider
class-specific recall, precision, F1 scores, and a confusion matrix when
evaluating the final model.

## 8. Why This Tutorial Uses single-pass Streams

The current dense minibatch iterator yields the expression matrix `X_batch`.
It does not return the corresponding cell identifiers or labels for randomized
batches.

A deterministic single-pass stream can therefore be aligned with a label
vector ordered by `filter_cell_id`.

```{important}
Do not combine a label vector ordered by `filter_cell_id` with
`pass_mode="multi-pass"`. Randomized minibatches will no longer correspond to
consecutive slices of that label vector.
```

A randomized supervised training interface should return the cell identifiers
or labels associated with every expression minibatch, conceptually:



In [ ]:
for cell_ids, X_batch in atlas.get_minibatch_dense(...):
    y_batch = lookup_labels(cell_ids)
    model.partial_fit(X_batch, y_batch)



Until such identifiers are available from the randomized iterator, repeated
single-pass traversal is the safer supervised pattern.

## 9. Inspect the Fitted Model

Check the fitted coefficient dimensions:



In [ ]:
print("Classes:", model.classes_)
print("Coefficient shape:", model.coef_.shape)
print("Intercept shape:", model.intercept_.shape)



For multiclass classification, the coefficient matrix contains one row per
class and one column per selected gene.

Convert encoded predictions back to their original labels with:



In [ ]:
predicted_names = label_encoder.inverse_transform(
    model.predict(X_batch)
)



Interpret model coefficients only when the feature identities, feature order,
and preprocessing representation have been retained.

## 10. Save the Model

Save the classifier and label encoder for later use:



In [ ]:
from pathlib import Path

import joblib

output_path = Path(
    "./results/logistic_regression_model.joblib"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

joblib.dump(
    {
        "model": model,
        "label_encoder": label_encoder,
    },
    output_path,
)

print(f"Saved model to {output_path}")



Also retain:

- the selected gene names in their exact read-index order;
- the expression field;
- normalization and scaling settings;
- the cell-selection definition;
- model hyperparameters;
- the scAtlasPy and scikit-learn versions.

The model cannot be applied safely to another Atlas unless its input features
match the training features exactly.

## Limitations of This Example

This tutorial demonstrates the data-access and alignment pattern rather than a
complete cell-type classification workflow.

It does not include:

- hyperparameter selection;
- class-imbalance correction;
- donor- or sample-aware validation;
- probability calibration;
- feature interpretation;
- automated early stopping;
- randomized supervised minibatches.

These decisions should be adapted to the biological question and intended
generalization setting.

## Next Steps

Continue with {doc}`train-pytorch-model-with-minibatches` for a neural-network
implementation of supervised minibatch training.

See {doc}`apply-model-to-full-atlas` to apply the fitted classifier to every
cell selected by a compatible read index.